# 🏥 Hybrid Clinical Decision Support System - Colab Training

This notebook trains a multi-class skin disease classifier using:
- **Melanoma** Detection
- **Eczema** Detection
- **Psoriasis** Detection  
- **Acne** Detection
- **Normal/Healthy Skin** Detection

**Hardware Requirements:**
- GPU: Recommended (Runtime → Change runtime type → GPU)
- RAM: 12.7 GB High-RAM runtime if available
- Disk: ~5GB for datasets

## 📦 Step 1: Setup Environment

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ No GPU detected. Training will be slower on CPU.")
    print("Consider: Runtime → Change runtime type → Hardware accelerator → GPU")

In [ ]:
# Install required packages
!pip install -q timm pyyaml gradio kaggle requests tqdm scikit-learn pillow opencv-python shap

## 💾 Step 2: Mount Google Drive (Optional - for saving models)

In [ ]:
# Mount Google Drive to save trained models
from google.colab import drive
drive.mount('/content/drive')

# Create project directory in Drive
import os
project_dir = '/content/drive/MyDrive/clinical_diagnosis_system'
os.makedirs(project_dir, exist_ok=True)
print(f"✓ Project directory: {project_dir}")

## 📥 Step 3: Clone Project from GitHub (or upload files)

In [ ]:
# Option A: Clone from GitHub (if you have a repo)
# !git clone https://github.com/yourusername/your-repo.git
# %cd your-repo

# Option B: Upload project files manually
# Use Colab's file upload feature to upload your project zip

# Option C: Start fresh (download individual files)
# This notebook assumes you have the project structure

# For this example, we'll create minimal structure
import os
os.makedirs('src', exist_ok=True)
os.makedirs('config', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

print("✓ Directory structure created")

## 🔑 Step 4: Setup Kaggle API Credentials

In [ ]:
# Upload your kaggle.json file
# Download from: https://www.kaggle.com/settings → API → Create New API Token

from google.colab import files
import os

print("📤 Upload your kaggle.json file when prompted...")
uploaded = files.upload()

# Setup Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✓ Kaggle API configured!")

## 📊 Step 5: Download Datasets

In [ ]:
# Download skin disease datasets from Kaggle
import kaggle
from pathlib import Path

data_dir = Path('data')
raw_dir = data_dir / 'skin_lesions_raw'
raw_dir.mkdir(parents=True, exist_ok=True)

datasets = {
    'melanoma': 'hasnainjaved/melanoma-skin-cancer-dataset-of-10000-images',
    'eczema': 'adityush/eczema2',
    'psoriasis': 'pallapurajkumar/psoriasis-skin-dataset',
    'acne': 'tiswan14/acne-dataset-image',
    'normal': 'shakyadissanayake/oily-dry-and-normal-skin-types-dataset'
}

print("📥 Downloading datasets (this may take 10-20 minutes)...\n")

for name, dataset_id in datasets.items():
    output_path = raw_dir / name
    output_path.mkdir(exist_ok=True)
    
    print(f"Downloading {name}...")
    try:
        kaggle.api.dataset_download_files(
            dataset_id,
            path=str(output_path),
            unzip=True,
            quiet=False
        )
        print(f"✓ {name} downloaded\n")
    except Exception as e:
        print(f"✗ Error downloading {name}: {e}\n")

print("✓ All datasets downloaded!")

## 🔧 Step 6: Normalize Dataset Structure

In [ ]:
# Normalize datasets into unified train/val/test structure
import shutil
from sklearn.model_selection import train_test_split
import random

def normalize_datasets():
    """Normalize all disease datasets into unified structure"""
    raw_dir = Path('data/skin_lesions_raw')
    output_dir = Path('data/skin_lesions')
    
    disease_mapping = {
        'melanoma': 'Melanoma Skin Cancer Nevi and Moles',
        'eczema': 'Eczema Photos',
        'psoriasis': 'Psoriasis pictures Lichen Planus and related diseases',
        'acne': 'Acne and Rosacea Photos',
        'normal': 'Normal Healthy Skin'
    }
    
    for split in ['train', 'val', 'test']:
        (output_dir / split).mkdir(parents=True, exist_ok=True)
    
    for short_name, full_name in disease_mapping.items():
        disease_dir = raw_dir / short_name
        
        if not disease_dir.exists():
            print(f"⚠️ {short_name} directory not found")
            continue
        
        print(f"\nProcessing {short_name}...")
        
        # Collect all images
        image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
        all_images = []
        for ext in image_extensions:
            all_images.extend(disease_dir.rglob(f'*{ext}'))
            all_images.extend(disease_dir.rglob(f'*{ext.upper()}'))
        
        if not all_images:
            print(f"  No images found!")
            continue
        
        print(f"  Found {len(all_images)} images")
        
        # Create splits: 70% train, 10% val, 20% test
        random.seed(42)
        random.shuffle(all_images)
        
        train_val, test = train_test_split(all_images, test_size=0.2, random_state=42)
        train, val = train_test_split(train_val, test_size=0.125, random_state=42)
        
        # Copy images to output directories
        for split_name, split_images in [('train', train), ('val', val), ('test', test)]:
            target_dir = output_dir / split_name / full_name
            target_dir.mkdir(parents=True, exist_ok=True)
            
            for i, img_path in enumerate(split_images):
                dest_path = target_dir / f"{short_name}_{i}{img_path.suffix}"
                shutil.copy2(img_path, dest_path)
            
            print(f"    {split_name}: {len(split_images)} images")
    
    print("\n✓ Dataset normalization complete!")

normalize_datasets()

## 🎓 Step 7: Train the CNN Model

In [ ]:
# If you have the full project uploaded, use:
# !python train.py --train-cnn

# Otherwise, use this standalone training code:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from pathlib import Path
from tqdm import tqdm
import numpy as np

# Dataset class
class SkinLesionDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Prepare data
def prepare_data():
    data_dir = Path('data/skin_lesions')
    
    class_names = [
        'Melanoma Skin Cancer Nevi and Moles',
        'Eczema Photos',
        'Psoriasis pictures Lichen Planus and related diseases',
        'Acne and Rosacea Photos',
        'Normal Healthy Skin'
    ]
    
    class_to_idx = {name: i for i, name in enumerate(class_names)}
    
    train_paths, train_labels = [], []
    val_paths, val_labels = [], []
    
    for class_name in class_names:
        # Training data
        train_class_dir = data_dir / 'train' / class_name
        if train_class_dir.exists():
            for img_path in train_class_dir.glob('*.*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    train_paths.append(str(img_path))
                    train_labels.append(class_to_idx[class_name])
        
        # Validation data
        val_class_dir = data_dir / 'val' / class_name
        if val_class_dir.exists():
            for img_path in val_class_dir.glob('*.*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    val_paths.append(str(img_path))
                    val_labels.append(class_to_idx[class_name])
    
    print(f"Training samples: {len(train_paths)}")
    print(f"Validation samples: {len(val_paths)}")
    
    return train_paths, train_labels, val_paths, val_labels, class_names

train_paths, train_labels, val_paths, val_labels, class_names = prepare_data()

In [ ]:
# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets and dataloaders
train_dataset = SkinLesionDataset(train_paths, train_labels, train_transform)
val_dataset = SkinLesionDataset(val_paths, val_labels, val_transform)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("✓ Model initialized")

In [ ]:
# Training loop
def train_model(model, train_loader, val_loader, epochs=10):
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({'loss': train_loss/len(pbar), 'acc': 100.*train_correct/train_total})
        
        train_loss /= len(train_loader)
        train_acc = 100. * train_correct / train_total
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        val_loss /= len(val_loader)
        val_acc = 100. * val_correct / val_total
        
        print(f"\nEpoch {epoch+1}:")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'class_names': class_names
            }, 'models/cnn_skin_lesion.pth')
            print(f"  ✓ Best model saved (Val Acc: {val_acc:.2f}%)")
    
    return history

# Train for 10 epochs (increase for better results)
history = train_model(model, train_loader, val_loader, epochs=10)

## 📊 Step 8: Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy
ax2.plot(history['train_acc'], label='Train Accuracy')
ax2.plot(history['val_acc'], label='Val Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"\n🎯 Final Results:")
print(f"  Best Validation Accuracy: {max(history['val_acc']):.2f}%")
print(f"  Final Training Accuracy: {history['train_acc'][-1]:.2f}%")

## 💾 Step 9: Save Model to Google Drive

In [ ]:
# Copy trained model to Google Drive
!cp models/cnn_skin_lesion.pth /content/drive/MyDrive/clinical_diagnosis_system/

print("✓ Model saved to Google Drive!")
print("You can now download it and use it locally.")

## 🧪 Step 10: Test the Model

In [ ]:
# Test on a few validation images
from PIL import Image
import random

model.eval()

# Select random validation images
test_indices = random.sample(range(len(val_paths)), 5)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for idx, test_idx in enumerate(test_indices):
    img_path = val_paths[test_idx]
    true_label = val_labels[test_idx]
    
    # Load and display image
    img = Image.open(img_path).convert('RGB')
    axes[idx].imshow(img)
    
    # Predict
    img_tensor = val_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img_tensor)
        probabilities = torch.softmax(output, dim=1)[0]
        pred_label = output.argmax(1).item()
        confidence = probabilities[pred_label].item()
    
    axes[idx].set_title(f"True: {class_names[true_label][:15]}...\nPred: {class_names[pred_label][:15]}...\nConf: {confidence:.2%}")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## ✅ Summary

You've successfully:
1. ✓ Set up the environment
2. ✓ Downloaded 5 skin disease datasets from Kaggle (including normal/healthy skin)
3. ✓ Normalized the data structure
4. ✓ Trained an EfficientNet-B0 model
5. ✓ Visualized training results
6. ✓ Saved the model to Google Drive

### Next Steps:
- Download the model from Google Drive
- Integrate it into your local project
- Run the web interface: `python app.py`
- Fine-tune with more epochs for better accuracy

### Tips for Better Results:
- Train for 20-50 epochs (this notebook uses 10 for speed)
- Use learning rate scheduling
- Add test-time augmentation
- Try ensemble methods with multiple models